# RF seed-6 diagnostic

The full main run completed all 420 fits. All 90 headline weighted-F1 values,
360 corresponding per-class F1 values and the saved feature-selection fields
match their original records. Four strict-comparison differences remain in
RF seed 6: one MI-sweep score and three two-packet-footprint C3 metrics.

This notebook collects a focused diagnostic using **two Random Forest fits**
on seed 6. It checks whether selected inputs differ across footprints,
whether repeated predictions change on a fixed forest, whether learned trees
match across fits, and how prediction threading and tree-summation order
relate to the observed scores.

1. Select **Runtime → Run all**.
2. Upload the updated **PQC_IDS_Reproduction_Working_Copy.zip** supplied with
   this notebook and connect the Drive account containing your CSV files.
3. Return **PQC_IDS_RF_Seed6_Diagnostic_Results.zip**, including any failure logs.

The original main run and its `needs_review` status are preserved. This
notebook collects diagnostic evidence; it does not choose the better score,
change a reproduction tolerance, replace historical results, or rerun the
420-fit experiment. It uses the same declared package versions as that run.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_rf6_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one updated working-copy ZIP.")
archive_path = SESSION / "working_copy.zip"
archive_path.write_bytes(uploaded[zip_names[0]])
del uploaded
extracted = SESSION / "package"
with zipfile.ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError("Unexpected archive path: " + info.filename)
    archive.extractall(extracted)
manifest_paths = list(extracted.glob("*/package_manifest.json"))
if len(manifest_paths) != 1:
    raise ValueError("Expected one working-copy package manifest.")
PACKAGE = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text())
listed = set()
for item in manifest["files"]:
    path = (PACKAGE / item["path"]).resolve()
    if not path.is_relative_to(PACKAGE.resolve()):
        raise ValueError("Invalid manifest path.")
    if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
        raise ValueError("Package file does not match the manifest: " + item["path"])
    listed.add(item["path"])
actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
    raise ValueError("The archive contains unlisted files or duplicate manifest records.")
if not (PACKAGE / "scripts/diagnose_rf_seed6.py").is_file():
    raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
print("Working-copy files verified:", len(listed))
print("Run identifier:", RUN_ID)

## Connect the verified input files

The default folder is `/content/drive/MyDrive/cicids2018`. Edit `DATA_DIR`
only if the files moved. The runner checks their hashes, the full prepared
matrix, seed-6 split indices and selected-feature order before fitting.

In [ ]:
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/pqc_ids_reproduction_checks") / RUN_ID
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=False)
RUN_OUTPUT = DRIVE_OUTPUT / "results"
TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
SUMMARY = {
    "run_id": RUN_ID,
    "status": "preparing_environment",
    "colab_kernel_python": platform.python_version(),
    "package_sha256": sha256(archive_path),
    "package_manifest_sha256": sha256(PACKAGE / "package_manifest.json"),
    "data_directory": str(DATA_DIR),
    "result_directory": str(DRIVE_OUTPUT),
    "scope": "Two RF seed-6 diagnostic fits; original main comparison remains needs_review",
    "target": TARGET,
}

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
shutil.copy2(PACKAGE / "environment/main_known_versions.json", DRIVE_OUTPUT / "main_known_versions.json")
shutil.copy2(PACKAGE / "environment/main_requirements.txt", DRIVE_OUTPUT / "main_requirements.txt")
print("Results will be saved to:", DRIVE_OUTPUT)
DIAGNOSTIC_PLAN = json.loads((PACKAGE / "environment/rf_seed6_diagnostic.json").read_text())
SUMMARY["diagnostic_plan"] = DIAGNOSTIC_PLAN
shutil.copy2(PACKAGE / "environment/rf_seed6_diagnostic.json", DRIVE_OUTPUT / "rf_seed6_diagnostic.json")
save_summary()

## Restore the same declared environment

Python 3.12.13 and the 21 pinned packages match the setup used for the returned
full main run. This creates a separate interpreter while your notebook kernel
keeps its own packages. It remains a documented new-run environment rather
than a complete reconstruction of the historical July runtime.

In [ ]:
RUN_ENV = os.environ.copy()
for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
    RUN_ENV.pop(name, None)
RUN_ENV["PYTHONNOUSERSITE"] = "1"
RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

def run_logged(command, log_name, env):
    with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
        log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
        log.flush()
        with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                              text=True, bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return process.wait()

ENVIRONMENT_READY = False
VENV = SESSION / "rf_diagnostic_environment"
FIT_PYTHON = VENV / "bin/python"
try:
    tooling = SESSION / "uv_tooling"
    bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                 "--no-deps", "uv==0.12.15"]
    if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
        raise RuntimeError("Could not install the pinned uv tool.")
    uv_env = RUN_ENV.copy()
    uv_env["PYTHONPATH"] = str(tooling)
    uv = [sys.executable, "-m", "uv"]
    create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
    if run_logged(create, "02_python_setup.log", uv_env) != 0:
        raise RuntimeError("Could not create the requested Python runtime.")
    install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                    "-r", PACKAGE / "environment/main_requirements.txt"]
    if run_logged(install, "03_packages_install.log", uv_env) != 0:
        raise RuntimeError("Could not install the declared package versions.")
    ENVIRONMENT_READY = True
    SUMMARY["status"] = "environment_installed"
except Exception as error:
    SUMMARY["status"] = "environment_setup_failed"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
    print("Setup failed:", error)
finally:
    save_summary()
print("Environment ready:", ENVIRONMENT_READY)

## Collect the fixed diagnostic plan

Both forests use the original RF constructor: 200 trees, model seed 42 and
`n_jobs=-1` during fitting. Feature selection is read from the matching seed-6
records, without tuning it on these diagnostic outcomes.

For **each fitted forest**, the runner collects:

- 20 repeated probability calls with the original prediction setting;
- one two-packet-footprint call at that setting;
- three calls with prediction restricted to one worker;
- one two-packet-footprint call with one worker;
- forward- and reverse-order sums of the same trees' probabilities.

This makes **54 captured probability calls across two fits**. Each call's
class labels and metrics are calculated from that same captured probability
array. The predictor uses the class with the highest mean tree probability;
`n_jobs` controls parallel work. [scikit-learn 1.6 documentation](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

The one-worker and reversed-order calculations are diagnostic controls.
They do not replace the archived experiment. Parallel aggregation near a tied
class score is a hypothesis to test, not an established explanation.

In [ ]:
if ENVIRONMENT_READY:
    try:
        SUMMARY["status"] = "running_rf_seed6_diagnostic"
        save_summary()
        command = [FIT_PYTHON, "-u", PACKAGE / "scripts/diagnose_rf_seed6.py",
                   "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                   "--cache-dir", SESSION / "input_cache"]
        returncode = run_logged(command, "04_rf_seed6_diagnostic.log", RUN_ENV)
        SUMMARY["diagnostic_returncode"] = returncode
        status_path = RUN_OUTPUT / "diagnostic_status.json"
        if status_path.is_file():
            diagnostic = json.loads(status_path.read_text())
            SUMMARY["status"] = diagnostic["status"]
            SUMMARY["fits_completed"] = diagnostic.get("fits_completed", 0)
            SUMMARY["prediction_calls_completed"] = diagnostic.get("prediction_calls_completed", 0)
            if SUMMARY["status"] == "diagnostic_completed":
                expected_calls = DIAGNOSTIC_PLAN["fits"] * (DIAGNOSTIC_PLAN["native_repetitions_per_fit"]
                                 + DIAGNOSTIC_PLAN["serial_repetitions_per_fit"] + 4)
                if returncode != 0 or SUMMARY["fits_completed"] != DIAGNOSTIC_PLAN["fits"] or SUMMARY["prediction_calls_completed"] != expected_calls:
                    SUMMARY["status"] = "diagnostic_incomplete"
            if "observations" in diagnostic:
                print(json.dumps(diagnostic["observations"], indent=2))
        else:
            SUMMARY["status"] = "runner_failed_before_status"
        SUMMARY["original_main_comparison_status"] = "needs_review"
        print("\nDiagnostic status:", SUMMARY["status"])
    except Exception as error:
        SUMMARY["status"] = "runner_exception"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "runner_traceback.txt").write_text(traceback.format_exc())
        print("Diagnostic failed:", error)
    finally:
        save_summary()
else:
    print("Diagnostic skipped because the declared environment was not installed.")

## Export the observations

The ZIP includes the complete call table, prediction arrays, selected-matrix
checks, learned-tree signatures, and probabilities and tree contributions for
records near a class tie or with changing predictions. It also contains
representative full probability arrays, the runtime, warnings and logs.
Raw CSV inputs and installation caches are excluded.

Run this cell even if a diagnostic step fails. The ZIP is also saved to the
new Drive result folder. Return **PQC_IDS_RF_Seed6_Diagnostic_Results.zip**.

In [ ]:
SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
save_summary()
result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*")
                      if p.is_file() and p.name not in ("PQC_IDS_RF_Seed6_Diagnostic_Results.zip", "result_manifest.json"))
manifest = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
             "sha256": sha256(p)} for p in result_files]
manifest_file = DRIVE_OUTPUT / "result_manifest.json"
manifest_file.write_text(json.dumps({"run_id": RUN_ID, "files": manifest}, indent=2) + "\n")
result_files.append(manifest_file)
result_zip = SESSION / "PQC_IDS_RF_Seed6_Diagnostic_Results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
with zipfile.ZipFile(result_zip) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("The result archive failed its integrity check.")
saved_zip = DRIVE_OUTPUT / result_zip.name
shutil.copy2(result_zip, saved_zip)
print("Final status:", SUMMARY["status"])
print("Results saved:", saved_zip)
print("ZIP SHA-256:", sha256(result_zip))
files.download(str(result_zip))